# cancer-recon-apoptosis — Step 2: Cancer-restricted target discovery (Colab)

**Goal:** a ranked shortlist of receptors enriched on CANCER cells vs matched NORMAL cells that participate in cell-cell communication — feeds Step 3 (specificity audit) and Step 4 (reward).

**Engine:** CELLxGENE Census (data) → **LIANA+** (Python; runs the CellChat method + CellPhoneDB/NATMI/… + a consensus). No R/Bioconductor.

**Runtime:** CPU is fine. For big pulls use **Runtime → Change runtime type → High-RAM**. **No GPU needed.**

**Run log:** every script run is teed to `runs/logs/step2_<UTC>_<gitSHA>.log` (stamped with the exact commit) and auto-downloaded by the last cell — share that file instead of copy-pasting. The commit SHA in the filename links the output to the code version that produced it.

**Data-first discipline:** run **Cell 3 (explore) FIRST** and read its output — it prints the real `disease` strings, cell counts, and whether malignant cells are annotated. Only then run Cell 4 (fetch).

**Honest scope:** finds cancer-*enriched* receptors (ranked), not cancer-*exclusive* ones. Specificity is enforced later by the ΔΔG-vs-normal-homolog reward.

## 1. Clone / pull repo (bootstrap)

In [ ]:
#@title Cell 1 — clone / pull repo (idempotent)
print('[CELL 1] ▶ clone or pull repo')
import os
from pathlib import Path
REPO_URL = 'https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git'
REPO_DIR = Path('/content/cancer-recon-apoptosis')
if REPO_DIR.exists():
    print('  pulling latest')
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print('  cwd =', os.getcwd())
assert (REPO_DIR / 'scripts' / '03_census_fetch.py').exists(), 'step-2 script missing'
assert (REPO_DIR / 'scripts' / 'runlog.py').exists(), 'runlog missing'
print('[CELL 1] ✓ done')

## 2. Start run log + install deps

Creates the session log (`runs/logs/step2_<UTC>_<sha>.log`) and tees the install into it. First install ~2-3 min.

In [ ]:
#@title Cell 2 — start run log + install (cellxgene-census / scanpy / liana)
import sys
from scripts.runlog import new_log, run_logged, finalize
RUNLOG = new_log('step2', repo_dir='.')      # one log per session; re-run to start fresh
import importlib.util
need = [m for m in ('cellxgene_census', 'scanpy', 'liana') if importlib.util.find_spec(m) is None]
if need:
    run_logged([sys.executable, '-m', 'pip', 'install', '-q', 'cellxgene-census', 'scanpy', 'liana'],
               RUNLOG, label='pip install cellxgene-census scanpy liana')
else:
    print('  deps already installed')
import cellxgene_census, scanpy
print('  cellxgene_census', cellxgene_census.__version__, '| scanpy', scanpy.__version__)
print('[CELL 2] ✓ done')

## 3. EXPLORE — look at the data first

Prints per tissue: total cells, available `disease` values (use the exact strings), top `cell_type` values, and whether `malignant cell` / `epithelial cell` are annotated. **Read this before fetching.** If the disease strings in `TARGETS` (top of `scripts/03_census_fetch.py`) don't match, fix them.

In [ ]:
#@title Cell 3 — explore Census metadata (cheap, ~1-2 min)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG = globals().get('RUNLOG') or new_log('step2', repo_dir='.')
print('[CELL 3] ▶ explore')
rc = run_logged([sys.executable, '-u', 'scripts/03_census_fetch.py', 'explore'], RUNLOG)
print(f'[CELL 3] exit={rc}', '✓' if rc==0 else '✗ see log above')

## 4. FETCH — pull tumour + normal AnnData per target

Capped + subsampled (default 20k cells/condition) → `data/cellxgene/<label>__<tumour|normal>.h5ad`. Idempotent. If a target reports **0 cells**, fix the disease string in `TARGETS` using Cell 3's output and re-run.

In [ ]:
#@title Cell 4 — fetch tumour + normal cells (minutes)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG = globals().get('RUNLOG') or new_log('step2', repo_dir='.')
print('[CELL 4] ▶ fetch')
rc = run_logged([sys.executable, '-u', 'scripts/03_census_fetch.py', 'fetch'], RUNLOG)
print(f'[CELL 4] exit={rc}', '✓' if rc==0 else '✗ see log above')
!ls -la data/cellxgene 2>/dev/null

## 5. Finalize — download the run log

Triggers a browser download of `runs/logs/step2_<UTC>_<sha>.log` (the full session output, commit-stamped). Share that file.

In [ ]:
#@title Cell 5 — finalize + download run log
from scripts.runlog import new_log, finalize
RUNLOG = globals().get('RUNLOG') or new_log('step2', repo_dir='.')
finalize(RUNLOG)            # prints path + downloads in Colab
!ls -la runs/logs 2>/dev/null

## 6. Next: Step 2b — LIANA+ communication inference

`scripts/04_liana_communication.py` (written after we confirm the data shape): runs LIANA+ on tumour vs normal AnnData and ranks receptors enriched in cancer. Share the downloaded run log so the analysis is designed from the real data.